In [1]:
import pandas as pd
import re
from Bio import SeqIO

In [ ]:
df=pd.read_table('repeats.tsv')
df['Start'] = df['Start'].str.replace(',','').astype(int)
df['End'] = df['End'].str.replace(',','').astype(int)
gene_locus={
    'Actin': (2143197,2145226),
    'α-Tubulin': (523332, 525367),
    'β-Tubulin': (1245400, 1247425),
    'Elongation factor 1': (1412453, 1414989),
    'PST130_P495001': (2674372, 2674809)
    }
ref_seq = 'GCF_021901695.1_Pst134E36_v1_pri_genomic.fna'
records = {r.id: str(r.seq) for r in SeqIO.parse(ref_seq, "fasta")}

In [30]:
def repeat_count(repeat_str):
    match = re.search(r'\((\D+)\)', str(repeat_str))
    return match.group(1) if match else None

In [36]:
def enrichment_count(repeat_str, seq):
    match = re.search(r'(\D+)-rich', str(repeat_str))
    nt = match.group(1) if match else None
    if len(nt) > 1:
        nt1 = nt[0]
        nt2 = nt[1]
        nt_count = 0
        for i in range(len(seq)):
            nt_count += seq[i].upper().count(nt1) if nt1 else 0
            nt_count += seq[i].upper().count(nt2) if nt2 else 0
    else:
        nt_count = 0
        for i in range(len(seq)):
            nt_count += seq[i].upper().count(nt) if nt else 0
    nt_pct = (nt_count / len(seq)) * 100 if len(seq) > 0 else 0
    return nt, nt_pct

In [ ]:
for row in df.itertuples():
    for gene, (start, end) in gene_locus.items():
        if gene == row.Gene:
            if row.Start <= start:
                df.at[row.Index, 'stream'] = 'Upstream'
                df.at[row.Index, 'proximity_to_CDS'] = start - row.Start
                if df.at[row.Index, 'proximity_to_CDS'] <= 2000:
                    df.at[row.Index, 'inside_promoter'] = 'YES'
                    promoter_df = df[df['inside_promoter'] == 'YES']
                else:
                    df.at[row.Index, 'inside_promoter'] = 'NO'

            else:
                df.at[row.Index, 'stream'] = 'Downstream'
                df.at[row.Index, 'proximity_to_CDS'] = row.End - start
                df.at[row.Index, 'inside_promoter'] = 'NO'

    repeat = repeat_count(row.Repeats)
    if repeat:
        count = len(repeat)
        df.at[row.Index, 'repeat_info'] = round((row.End - row.Start) / int(count))
        df.at[row.Index, 'new_repeats'] = f'({repeat}){round((row.End - row.Start) / int(count))}'
    elif re.search(r'\w+-rich', str(row.Repeats)):
        chrom = row.Location
        start = row.Start
        end = row.End

        nt, nt_pct = enrichment_count(row.Repeats, records[chrom][start:end])
        df.at[row.Index, 'repeat_info'] = round(nt_pct)
        df.at[row.Index, 'new_repeats'] = f'{nt}-rich ({round(nt_pct)}%)'

    else:
        df.at[row.Index, 'repeat_info'] = None

promoter_df

,Gene,Location,Start,End,Repeats,stream,proximity_to_CDS,repeat_info,inside_promoter,new_repeats
2,Actin,NC_063041.1,2142734,2142772,(ATG)n,Upstream,463.0,13.0,YES,(ATG)13
8,α-Tubulin,NC_063035.1,523060,523087,(A)n,Upstream,272.0,27.0,YES,(A)27
19,β-Tubulin,NC_063031.1,1244767,1244789,(ATC)n,Upstream,633.0,7.0,YES,(ATC)7
20,β-Tubulin,NC_063031.1,1245040,1245084,(TCAGTC)n,Upstream,360.0,7.0,YES,(TCAGTC)7
21,β-Tubulin,NC_063031.1,1245067,1245164,(TCTCCA)n,Upstream,333.0,16.0,YES,(TCTCCA)16
34,Elongation factor 1,NC_063031.1,1411942,1411966,(C)n,Upstream,511.0,24.0,YES,(C)24


In [44]:
df[df['Gene'] == 'PST130_P495001']
df.to_csv('repeats_updated.csv', index=False)
promoter_df.to_csv('repeats_in_promoter.csv', index=False)